# 🐘 SQL con PostgreSQL + Python

Notebook de referencia para aprender y repasar SQL desde cero.
Cada sección tiene teoría, ejemplos ejecutables y ejercicios prácticos.

**Motor:** PostgreSQL 16 (Docker)  
**Driver:** psycopg2  
**Schema:** e-commerce (clientes, productos, categorías, pedidos)

---
> ⚠️ **Ejecutar esta primera sección antes de cualquier otra celda**

## 0. Conexión a la base de datos

Antes de ejecutar cualquier query necesitamos conectarnos a PostgreSQL.
Usamos `psycopg2`, el driver más usado en proyectos Python profesionales.

`RealDictCursor` hace que cada fila devuelta sea un diccionario
en lugar de una tupla — más legible y fácil de trabajar.

## 0. Conexión a la base de datos

`psycopg` es el driver oficial de PostgreSQL para Python (v3, la versión moderna).
Antes de ejecutar cualquier query necesitamos importarlo y definir cómo conectarnos.

- `psycopg` — el driver en sí, el que "habla" con PostgreSQL
- `dict_row` — un adaptador que hace que cada fila devuelta sea un diccionario
  (`{"id": 1, "nombre": "Ana"}`) en lugar de una tupla (`(1, "Ana")`)

`CONN_STR` es la cadena de conexión: un string con todos los datos necesarios
para que psycopg encuentre y se autentique con la base de datos.

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `host` | `localhost` | La base corre en esta misma máquina (dentro de Docker) |
| `port` | `5433` | Puerto donde Docker expone PostgreSQL |
| `dbname` | `ecommerce` | Nombre de la base de datos a usar |
| `user` | `alumno` | Usuario de PostgreSQL |
| `password` | `alumno123` | Contraseña del usuario |

In [2]:
# sql-aprendizaje/sql_psycopg.ipynb
import psycopg
from psycopg.rows import dict_row

CONN_STR = "host=localhost port=5433 dbname=ecommerce user=alumno password=alumno123"

## Funciones de utilidad

En lugar de repetir el código de conexión en cada celda, definimos dos funciones
reutilizables que usaremos a lo largo de todo el notebook.

### `obtener_conexion()`

Crea y devuelve una conexión a PostgreSQL con dos opciones importantes:

- `autocommit=True` — cada sentencia SQL se confirma automáticamente en la base
  de datos sin necesidad de llamar a `commit()` manualmente. Ideal para un notebook
  de consultas y aprendizaje; en una aplicación real se manejaría con transacciones explícitas.
- `row_factory=dict_row` — le dice a psycopg que devuelva cada fila como diccionario
  en lugar de tupla, así podemos acceder a los datos por nombre de columna: `fila["nombre"]`.

### `ejecutar_query(sql, params=None)`

Ejecuta cualquier sentencia SQL y devuelve los resultados.

- `conn` — la conexión activa a PostgreSQL. El bloque `with` garantiza que se cierre
  automáticamente al terminar, aunque ocurra un error.
- `cur` (cursor) — el objeto que envía las sentencias SQL al servidor y recibe los resultados.
  Análogo a un puntero dentro de la conexión.
- `sql` — la sentencia SQL a ejecutar, como string.
- `params` — valores a insertar en la query de forma segura (evita SQL injection).
  Se pasa como tupla o diccionario y psycopg los escapa automáticamente.
  Ejemplo: `ejecutar_query("SELECT * FROM clientes WHERE id = %s", (1,))`
- `fetchall()` — recupera todas las filas del resultado como lista. Si la sentencia
  no devuelve filas (un `INSERT` o `CREATE TABLE` por ejemplo), lanza una excepción
  que capturamos devolviendo una lista vacía.

In [3]:
def obtener_conexion():
    """Devuelve una conexion con autocommit y filas como diccionarios."""
    return psycopg.connect(CONN_STR, autocommit=True, row_factory=dict_row)

def ejecutar_query(sql, params=None):
    """Ejecuta una query y devuelve los resultados como lista de diccionarios."""
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            try:
                return cur.fetchall()
            except Exception:
                return []

### Verificación de la conexión

Antes de continuar, comprobamos que todo el stack está funcionando:
Docker corriendo, PostgreSQL levantado, credenciales correctas.

- `with obtener_conexion() as conn` — abre la conexión y la cierra automáticamente
  al salir del bloque, incluso si ocurre un error.
- `with conn.cursor() as cur` — abre un cursor dentro de esa conexión. Un cursor
  es el canal por donde viajan las sentencias SQL hacia el servidor y los resultados
  de vuelta.
- `cur.execute("SELECT version();")` — ejecuta una query nativa de PostgreSQL que
  devuelve la versión del servidor. Es el "hola mundo" de SQL.
- `fetchone()` — recupera solo la primera fila del resultado. A diferencia de
  `fetchall()`, no trae todo sino una sola fila. Como usamos `dict_row`, esa fila
  es un diccionario.
- `version["version"]` — accedemos al valor por nombre de columna. PostgreSQL
  devuelve la columna con el nombre `version`, entonces usamos esa clave.
- El bloque `try/except` captura cualquier error de conexión (Docker apagado,
  credenciales incorrectas, puerto equivocado) y lo muestra en lugar de romper
  el notebook.

Si la celda imprime la versión de PostgreSQL, el entorno está listo.

In [4]:
try:
    with obtener_conexion() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT version();")
            version = cur.fetchone()
    print("Conexion exitosa a PostgreSQL")
    print(version["version"])
except Exception as e:
    print(f"Error de conexion: {e}")

Conexion exitosa a PostgreSQL
PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 01. Fundamentos — DDL, DML y tipos de datos

SQL se divide en tres grandes grupos de sentencias según su propósito:

| Grupo | Nombre completo | Para qué sirve | Ejemplos |
|-------|----------------|----------------|---------|
| DDL | Data Definition Language | Definir la estructura de la base | `CREATE`, `ALTER`, `DROP` |
| DML | Data Manipulation Language | Manipular los datos dentro de las tablas | `INSERT`, `SELECT`, `UPDATE`, `DELETE` |
| DCL | Data Control Language | Controlar permisos y accesos | `GRANT`, `REVOKE` |

En esta sección trabajamos DDL: vamos a crear las tablas del schema e-commerce desde cero.

### Tipos de datos en PostgreSQL

Antes de crear una tabla hay que decidir qué tipo de dato va en cada columna.
Los más usados:

| Tipo | Descripción | Ejemplo |
|------|-------------|---------|
| `SERIAL` | Entero autoincremental, típico para IDs | `1, 2, 3, ...` |
| `INTEGER` | Número entero | `42` |
| `NUMERIC(p, s)` | Decimal de precisión fija. `p` = dígitos totales, `s` = decimales | `NUMERIC(10, 2)` → `12345678.90` |
| `VARCHAR(n)` | Texto de longitud variable con límite `n` | `'Ana García'` |
| `TEXT` | Texto sin límite de longitud | descripción larga |
| `BOOLEAN` | Verdadero o falso | `TRUE`, `FALSE` |
| `DATE` | Fecha sin hora | `2024-01-15` |
| `TIMESTAMP` | Fecha con hora | `2024-01-15 10:30:00` |

`SERIAL` es un shorthand de PostgreSQL: crea un entero con una secuencia automática.
En PostgreSQL moderno también existe `GENERATED ALWAYS AS IDENTITY`, pero `SERIAL`
sigue siendo el más común en la práctica.

### Creando el schema e-commerce

Vamos a crear 4 tablas que representan un sistema de comercio electrónico real.
Por ahora sin constraints entre tablas (eso va en la sección 02), solo la estructura básica.

El orden importa: si una tabla referencia a otra, la referenciada debe existir primero.
En este caso creamos `categorias` antes que `productos` porque productos la va a necesitar.

In [5]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS categorias (
                id        SERIAL PRIMARY KEY,
                nombre    VARCHAR(100) NOT NULL,
                activa    BOOLEAN DEFAULT TRUE
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS productos (
                id              SERIAL PRIMARY KEY,
                nombre          VARCHAR(200) NOT NULL,
                precio          NUMERIC(10, 2) NOT NULL,
                stock           INTEGER DEFAULT 0,
                id_categoria    INTEGER,
                creado_en       TIMESTAMP DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS clientes (
                id          SERIAL PRIMARY KEY,
                nombre      VARCHAR(150) NOT NULL,
                email       VARCHAR(200),
                creado_en   DATE DEFAULT CURRENT_DATE
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS pedidos (
                id          SERIAL PRIMARY KEY,
                id_cliente  INTEGER,
                total       NUMERIC(10, 2),
                estado      VARCHAR(50) DEFAULT 'pendiente',
                creado_en   TIMESTAMP DEFAULT NOW()
            );
        """)

print("Tablas creadas correctamente")

Tablas creadas correctamente


### Verificar las tablas creadas

`information_schema.tables` es una vista del sistema que PostgreSQL mantiene
automáticamente con metadata de todas las tablas existentes.
Filtramos por `table_schema = 'public'` para ver solo las nuestras,
excluyendo las tablas internas del sistema.

In [6]:
tablas = ejecutar_query("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

for tabla in tablas:
    print(tabla["table_name"])

categorias
clientes
pedidos
productos


## 02. Constraints — Restricciones de integridad

Un constraint es una regla que PostgreSQL aplica automáticamente sobre los datos.
Si una operación viola una regla, el motor la rechaza y lanza un error.
Sirven para garantizar que la base de datos nunca tenga datos inválidos,
sin depender de que la aplicación lo valide correctamente.

| Constraint | Para qué sirve |
|------------|----------------|
| `PRIMARY KEY` | Identifica de forma única cada fila. Implica `NOT NULL` + `UNIQUE` |
| `FOREIGN KEY` | Vincula una columna con la PK de otra tabla. Garantiza integridad referencial |
| `UNIQUE` | No permite valores duplicados en esa columna |
| `NOT NULL` | La columna no puede quedar vacía |
| `DEFAULT` | Valor que toma la columna si no se especifica uno al insertar |
| `CHECK` | Valida que el valor cumpla una condición lógica arbitraria |

### Recreando el schema con constraints completos

En la sección anterior creamos las tablas sin relaciones entre ellas.
Ahora las eliminamos y las volvemos a crear con todos los constraints.

`DROP TABLE IF EXISTS ... CASCADE` elimina la tabla y cualquier Foreign Key
que otras tablas tengan hacia ella, evitando errores de dependencia.
El orden de eliminación es el inverso al de creación: primero las que
dependen de otras, después las independientes.

In [7]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            DROP TABLE IF EXISTS pedidos   CASCADE;
            DROP TABLE IF EXISTS productos CASCADE;
            DROP TABLE IF EXISTS clientes  CASCADE;
            DROP TABLE IF EXISTS categorias CASCADE;
        """)

print("Tablas eliminadas")

Tablas eliminadas


### PRIMARY KEY y NOT NULL

`PRIMARY KEY` es el identificador único de cada fila.
- Solo puede haber una por tabla
- No acepta `NULL`
- PostgreSQL crea automáticamente un índice sobre ella (las búsquedas por ID son rápidas)

`NOT NULL` garantiza que la columna siempre tenga un valor.
Sin este constraint, cualquier columna acepta `NULL` por defecto.

In [8]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE categorias (
                id      SERIAL PRIMARY KEY,
                nombre  VARCHAR(100) NOT NULL,
                activa  BOOLEAN DEFAULT TRUE
            );
        """)

print("Tabla categorias creada")

Tabla categorias creada


### FOREIGN KEY

Una Foreign Key (FK) vincula una columna de una tabla con la Primary Key de otra.
Garantiza integridad referencial: no se puede insertar un `id_categoria` en `productos`
que no exista en `categorias`.

`ON DELETE` define qué pasa con los productos si se elimina su categoría:
- `RESTRICT` — rechaza el DELETE si hay productos que la referencian (la más segura)
- `CASCADE` — elimina también todos los productos de esa categoría
- `SET NULL` — pone `NULL` en `id_categoria` de los productos huérfanos

In [9]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE productos (
                id            SERIAL PRIMARY KEY,
                nombre        VARCHAR(200) NOT NULL,
                precio        NUMERIC(10, 2) NOT NULL,
                stock         INTEGER DEFAULT 0,
                id_categoria  INTEGER REFERENCES categorias(id) ON DELETE RESTRICT,
                creado_en     TIMESTAMP DEFAULT NOW()
            );
        """)

print("Tabla productos creada")

Tabla productos creada


### UNIQUE y CHECK

`UNIQUE` impide duplicados en una columna. A diferencia de `PRIMARY KEY`,
puede haber varias columnas `UNIQUE` en una tabla, y sí acepta `NULL`
(PostgreSQL considera que dos `NULL` no son iguales entre sí).

`CHECK` valida una condición arbitraria antes de aceptar el valor.
Si la condición devuelve `FALSE`, PostgreSQL rechaza la operación.
Ejemplos útiles:
- Que el precio no sea negativo
- Que el stock no baje de cero  
- Que el estado solo pueda ser uno de un conjunto de valores

In [10]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE clientes (
                id         SERIAL PRIMARY KEY,
                nombre     VARCHAR(150) NOT NULL,
                email      VARCHAR(200) UNIQUE,
                creado_en  DATE DEFAULT CURRENT_DATE
            );
        """)

        cur.execute("""
            CREATE TABLE pedidos (
                id          SERIAL PRIMARY KEY,
                id_cliente  INTEGER REFERENCES clientes(id) ON DELETE RESTRICT,
                total       NUMERIC(10, 2) CHECK (total >= 0),
                estado      VARCHAR(50) DEFAULT 'pendiente'
                                CHECK (estado IN ('pendiente', 'pagado', 'enviado', 'cancelado')),
                creado_en   TIMESTAMP DEFAULT NOW()
            );
        """)

print("Tablas clientes y pedidos creadas")

Tablas clientes y pedidos creadas


### Verificar los constraints creados

`information_schema.table_constraints` lista todos los constraints de la base.
Filtramos por nuestras cuatro tablas para ver qué reglas quedaron activas.

In [11]:
constraints = ejecutar_query("""
    SELECT table_name, constraint_name, constraint_type
    FROM information_schema.table_constraints
    WHERE table_schema = 'public'
    ORDER BY table_name, constraint_type;
""")

for c in constraints:
    print(f"{c['table_name']:15} {c['constraint_type']:12} {c['constraint_name']}")

categorias      CHECK        2200_16521_2_not_null
categorias      CHECK        2200_16521_1_not_null
categorias      PRIMARY KEY  categorias_pkey
clientes        CHECK        2200_16543_1_not_null
clientes        CHECK        2200_16543_2_not_null
clientes        PRIMARY KEY  clientes_pkey
clientes        UNIQUE       clientes_email_key
pedidos         CHECK        2200_16553_1_not_null
pedidos         CHECK        pedidos_total_check
pedidos         CHECK        pedidos_estado_check
pedidos         FOREIGN KEY  pedidos_id_cliente_fkey
pedidos         PRIMARY KEY  pedidos_pkey
productos       CHECK        2200_16529_2_not_null
productos       CHECK        2200_16529_3_not_null
productos       CHECK        2200_16529_1_not_null
productos       FOREIGN KEY  productos_id_categoria_fkey
productos       PRIMARY KEY  productos_pkey


## 03. CRUD — INSERT, SELECT, UPDATE, DELETE

CRUD es el acrónimo de las cuatro operaciones básicas sobre datos:

| Operación | SQL | Descripción |
|-----------|-----|-------------|
| Create | `INSERT` | Agregar filas nuevas a una tabla |
| Read | `SELECT` | Consultar y recuperar datos |
| Update | `UPDATE` | Modificar filas existentes |
| Delete | `DELETE` | Eliminar filas |

Todas pertenecen al grupo DML (Data Manipulation Language).
A partir de acá empezamos a poblar el schema e-commerce con datos reales
y a consultarlos de distintas formas.

### INSERT — Agregar datos

Sintaxis básica:
```sql
INSERT INTO tabla (columna1, columna2) VALUES (valor1, valor2);
```

Puntos importantes:
- Las columnas con `DEFAULT` o `SERIAL` pueden omitirse; PostgreSQL las completa solo.
- Para insertar múltiples filas en una sola sentencia se encadenan los `VALUES`.
- psycopg usa `%s` como placeholder para parámetros seguros — nunca se concatena
  el valor directamente en el string SQL, eso abre vulnerabilidades de SQL injection.

In [12]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        # Insertar categorias
        cur.executemany(
            "INSERT INTO categorias (nombre) VALUES (%s)",
            [("Electronica",), ("Ropa",), ("Hogar",), ("Deportes",)]
        )

        # Insertar productos con parametros nombrados
        productos = [
            ("Laptop 15\"",   1299.99, 10, 1),
            ("Auriculares",    89.99, 35, 1),
            ("Camiseta M",     19.99, 100, 2),
            ("Zapatillas 42",  79.99, 50, 4),
            ("Lampara LED",    34.99, 60, 3),
        ]
        cur.executemany(
            "INSERT INTO productos (nombre, precio, stock, id_categoria) VALUES (%s, %s, %s, %s)",
            productos
        )

        # Insertar clientes
        clientes = [
            ("Ana Torres",    "ana@email.com"),
            ("Luis Gomez",    "luis@email.com"),
            ("Maria Lopez",   "maria@email.com"),
            ("Carlos Ruiz",   "carlos@email.com"),
        ]
        cur.executemany(
            "INSERT INTO clientes (nombre, email) VALUES (%s, %s)",
            clientes
        )

        # Insertar pedidos
        pedidos = [
            (1, 1389.98, "pagado"),
            (2,   79.99, "enviado"),
            (1,   34.99, "pendiente"),
            (3,   89.99, "pagado"),
        ]
        cur.executemany(
            "INSERT INTO pedidos (id_cliente, total, estado) VALUES (%s, %s, %s)",
            pedidos
        )

print("Datos insertados correctamente")

Datos insertados correctamente


### SELECT — Consultar datos

`SELECT` es la sentencia más usada en SQL. Su estructura básica:

```sql
SELECT columnas
FROM tabla
WHERE condicion
ORDER BY columna;
```

- `*` selecciona todas las columnas. En producción conviene listar solo las necesarias.
- `WHERE` filtra filas — sin él se devuelven todas.
- `ORDER BY` ordena el resultado. `ASC` es ascendente (default), `DESC` descendente.
- `LIMIT` restringe la cantidad de filas devueltas, útil para explorar tablas grandes.

In [13]:
# Todos los productos
productos = ejecutar_query("SELECT * FROM productos ORDER BY precio DESC;")
for p in productos:
    print(f"{p['nombre']:20} ${p['precio']}")

print()

# Solo nombre y email de clientes, ordenados alfabeticamente
clientes = ejecutar_query("SELECT nombre, email FROM clientes ORDER BY nombre ASC;")
for c in clientes:
    print(f"{c['nombre']:20} {c['email']}")

print()

# Los 3 productos mas baratos
baratos = ejecutar_query("SELECT nombre, precio FROM productos ORDER BY precio ASC LIMIT 3;")
for p in baratos:
    print(f"{p['nombre']:20} ${p['precio']}")

Laptop 15"           $1299.99
Auriculares          $89.99
Zapatillas 42        $79.99
Lampara LED          $34.99
Camiseta M           $19.99

Ana Torres           ana@email.com
Carlos Ruiz          carlos@email.com
Luis Gomez           luis@email.com
Maria Lopez          maria@email.com

Camiseta M           $19.99
Lampara LED          $34.99
Zapatillas 42        $79.99


### UPDATE — Modificar datos

```sql
UPDATE tabla
SET columna1 = valor1, columna2 = valor2
WHERE condicion;
```

⚠️ El `WHERE` es crítico: sin él se actualizan **todas** las filas de la tabla.
Siempre verificar la condición antes de ejecutar.

`RETURNING` es una extensión de PostgreSQL que devuelve las filas afectadas
después del UPDATE, útil para confirmar qué cambió sin hacer una query adicional.

In [14]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        # Actualizar precio de un producto
        cur.execute("""
            UPDATE productos
            SET precio = 1199.99
            WHERE nombre = 'Laptop 15"'
            RETURNING id, nombre, precio;
        """)
        actualizado = cur.fetchone()
        print(f"Actualizado: {actualizado['nombre']} → ${actualizado['precio']}")

        # Cambiar estado de un pedido
        cur.execute("""
            UPDATE pedidos
            SET estado = 'enviado'
            WHERE id = 3
            RETURNING id, estado;
        """)
        pedido = cur.fetchone()
        print(f"Pedido {pedido['id']} → {pedido['estado']}")

Actualizado: Laptop 15" → $1199.99
Pedido 3 → enviado


### DELETE — Eliminar datos

```sql
DELETE FROM tabla
WHERE condicion;
```

⚠️ Igual que `UPDATE`, sin `WHERE` elimina todas las filas.
A diferencia de `DROP TABLE`, `DELETE` elimina filas pero mantiene la tabla.

Alternativa para vaciar una tabla completamente: `TRUNCATE tabla` —
más rápido que `DELETE` sin `WHERE` porque no recorre fila por fila,
pero no dispara triggers y no se puede filtrar.

In [15]:
with obtener_conexion() as conn:
    with conn.cursor() as cur:

        # Insertar un cliente de prueba para luego eliminarlo
        cur.execute("""
            INSERT INTO clientes (nombre, email)
            VALUES ('Prueba Borrar', 'borrar@email.com')
            RETURNING id, nombre;
        """)
        nuevo = cur.fetchone()
        print(f"Insertado: id={nuevo['id']} nombre={nuevo['nombre']}")

        # Eliminar ese cliente
        cur.execute("""
            DELETE FROM clientes
            WHERE email = 'borrar@email.com'
            RETURNING id, nombre;
        """)
        eliminado = cur.fetchone()
        print(f"Eliminado: id={eliminado['id']} nombre={eliminado['nombre']}")

# Verificar que solo quedan los 4 clientes originales
clientes = ejecutar_query("SELECT COUNT(*) as total FROM clientes;")
print(f"Clientes en tabla: {clientes[0]['total']}")

Insertado: id=5 nombre=Prueba Borrar
Eliminado: id=5 nombre=Prueba Borrar
Clientes en tabla: 4


## 04. Filtros — WHERE, BETWEEN, IN, LIKE, IS NULL

`WHERE` es la cláusula que filtra qué filas devuelve o afecta una sentencia.
Sin ella, `SELECT`, `UPDATE` y `DELETE` operan sobre toda la tabla.

Los filtros se construyen con operadores que se pueden combinar con `AND` y `OR`:

| Operador | Descripción | Ejemplo |
|----------|-------------|---------|
| `=` | Igual | `estado = 'pagado'` |
| `<>` o `!=` | Distinto | `estado <> 'cancelado'` |
| `>` `<` `>=` `<=` | Comparación numérica o de fechas | `precio > 100` |
| `BETWEEN` | Rango inclusivo en ambos extremos | `precio BETWEEN 50 AND 200` |
| `IN` | Pertenece a un conjunto de valores | `estado IN ('pagado', 'enviado')` |
| `LIKE` | Coincidencia de patrón en texto | `nombre LIKE 'La%'` |
| `IS NULL` | La columna no tiene valor | `email IS NULL` |
| `IS NOT NULL` | La columna tiene valor | `email IS NOT NULL` |

### Operadores de comparación y lógicos

`AND` requiere que todas las condiciones sean verdaderas.
`OR` requiere que al menos una lo sea.
`NOT` invierte el resultado de una condición.

El orden de precedencia es `NOT` → `AND` → `OR`.
Cuando se mezclan, usar paréntesis para dejar la intención explícita.

In [16]:
# Productos con precio mayor a 50
print("--- Precio > 50 ---")
resultado = ejecutar_query("""
    SELECT nombre, precio
    FROM productos
    WHERE precio > 50
    ORDER BY precio;
""")
for r in resultado:
    print(f"{r['nombre']:20} ${r['precio']}")

# Productos baratos con stock alto
print("\n--- Precio < 50 AND stock > 50 ---")
resultado = ejecutar_query("""
    SELECT nombre, precio, stock
    FROM productos
    WHERE precio < 50 AND stock > 50
    ORDER BY stock DESC;
""")
for r in resultado:
    print(f"{r['nombre']:20} ${r['precio']} stock:{r['stock']}")

--- Precio > 50 ---
Zapatillas 42        $79.99
Auriculares          $89.99
Laptop 15"           $1199.99

--- Precio < 50 AND stock > 50 ---
Camiseta M           $19.99 stock:100
Lampara LED          $34.99 stock:60


### BETWEEN — Rango de valores

```sql
WHERE columna BETWEEN valor_minimo AND valor_maximo
```

Equivale a `columna >= minimo AND columna <= maximo` — ambos extremos son inclusivos.
Funciona con números, fechas y texto (orden alfabético).

`NOT BETWEEN` devuelve los valores fuera del rango.

In [17]:
# Productos en rango de precio
print("--- Precio BETWEEN 30 AND 100 ---")
resultado = ejecutar_query("""
    SELECT nombre, precio
    FROM productos
    WHERE precio BETWEEN 30 AND 100
    ORDER BY precio;
""")
for r in resultado:
    print(f"{r['nombre']:20} ${r['precio']}")

# Pedidos fuera de ese rango
print("\n--- Pedidos NOT BETWEEN 50 AND 500 ---")
resultado = ejecutar_query("""
    SELECT id, total, estado
    FROM pedidos
    WHERE total NOT BETWEEN 50 AND 500
    ORDER BY total;
""")
for r in resultado:
    print(f"Pedido {r['id']}  ${r['total']}  {r['estado']}")

--- Precio BETWEEN 30 AND 100 ---
Lampara LED          $34.99
Zapatillas 42        $79.99
Auriculares          $89.99

--- Pedidos NOT BETWEEN 50 AND 500 ---
Pedido 3  $34.99  enviado
Pedido 1  $1389.98  pagado


### IN — Conjunto de valores

```sql
WHERE columna IN (valor1, valor2, valor3)
```

Es un shorthand legible de múltiples condiciones `OR`.
Esta query:
```sql
WHERE estado IN ('pagado', 'enviado')
```
es equivalente a:
```sql
WHERE estado = 'pagado' OR estado = 'enviado'
```

`NOT IN` excluye los valores del conjunto.
Cuidado con `NOT IN` cuando el conjunto puede contener `NULL` —
en ese caso la condición entera devuelve `NULL` y no filtra como se espera.

In [18]:
# Pedidos pagados o enviados
print("--- Estado IN ('pagado', 'enviado') ---")
resultado = ejecutar_query("""
    SELECT id, total, estado
    FROM pedidos
    WHERE estado IN ('pagado', 'enviado')
    ORDER BY estado;
""")
for r in resultado:
    print(f"Pedido {r['id']}  ${r['total']}  {r['estado']}")

# Productos de categorias especificas
print("\n--- Categorias 1 y 4 ---")
resultado = ejecutar_query("""
    SELECT nombre, id_categoria
    FROM productos
    WHERE id_categoria IN (1, 4)
    ORDER BY id_categoria;
""")
for r in resultado:
    print(f"{r['nombre']:20} cat:{r['id_categoria']}")

--- Estado IN ('pagado', 'enviado') ---
Pedido 2  $79.99  enviado
Pedido 3  $34.99  enviado
Pedido 1  $1389.98  pagado
Pedido 4  $89.99  pagado

--- Categorias 1 y 4 ---
Auriculares          cat:1
Laptop 15"           cat:1
Zapatillas 42        cat:4


### LIKE — Coincidencia de patrones

```sql
WHERE columna LIKE 'patron'
```

Los caracteres especiales del patrón son:
- `%` — cualquier secuencia de caracteres (incluyendo ninguno)
- `_` — exactamente un carácter cualquiera

Ejemplos:
| Patrón | Coincide con |
|--------|-------------|
| `'La%'` | Todo lo que empieza con "La" |
| `'%LED'` | Todo lo que termina en "LED" |
| `'%a%'` | Todo lo que contiene "a" |
| `'_a%'` | Todo lo que tiene "a" en la segunda posición |

`ILIKE` es la versión case-insensitive de PostgreSQL (no existe en todos los motores SQL).

In [19]:
# Productos que empiezan con 'La'
print("--- LIKE 'La%' ---")
resultado = ejecutar_query("""
    SELECT nombre FROM productos
    WHERE nombre LIKE 'La%';
""")
for r in resultado:
    print(r['nombre'])

# Clientes cuyo email termina en .com
print("\n--- Email LIKE '%.com' ---")
resultado = ejecutar_query("""
    SELECT nombre, email FROM clientes
    WHERE email LIKE '%.com'
    ORDER BY nombre;
""")
for r in resultado:
    print(f"{r['nombre']:20} {r['email']}")

# Busqueda case-insensitive
print("\n--- Nombre ILIKE '%lopez%' ---")
resultado = ejecutar_query("""
    SELECT nombre FROM clientes
    WHERE nombre ILIKE '%lopez%';
""")
for r in resultado:
    print(r['nombre'])

--- LIKE 'La%' ---
Lampara LED
Laptop 15"

--- Email LIKE '%.com' ---
Ana Torres           ana@email.com
Carlos Ruiz          carlos@email.com
Luis Gomez           luis@email.com
Maria Lopez          maria@email.com

--- Nombre ILIKE '%lopez%' ---
Maria Lopez


### IS NULL / IS NOT NULL

`NULL` en SQL significa ausencia de valor — no es un string vacío ni un cero.
Por eso no se puede comparar con `=`: `columna = NULL` siempre devuelve `NULL`,
nunca `TRUE`. La única forma correcta es `IS NULL` o `IS NOT NULL`.

Casos típicos:
- Clientes que no proporcionaron email
- Productos sin categoría asignada
- Pedidos sin fecha de envío registrada

In [20]:
# Insertar un cliente sin email para el ejemplo
with obtener_conexion() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO clientes (nombre)
            VALUES ('Sin Email')
            RETURNING id, nombre, email;
        """)
        nuevo = cur.fetchone()
        print(f"Insertado: {nuevo['nombre']} — email: {nuevo['email']}")

# Clientes sin email
print("\n--- Email IS NULL ---")
resultado = ejecutar_query("""
    SELECT nombre, email FROM clientes
    WHERE email IS NULL;
""")
for r in resultado:
    print(f"{r['nombre']} — sin email")

# Clientes con email
print("\n--- Email IS NOT NULL ---")
resultado = ejecutar_query("""
    SELECT nombre, email FROM clientes
    WHERE email IS NOT NULL
    ORDER BY nombre;
""")
for r in resultado:
    print(f"{r['nombre']:20} {r['email']}")

Insertado: Sin Email — email: None

--- Email IS NULL ---
Sin Email — sin email

--- Email IS NOT NULL ---
Ana Torres           ana@email.com
Carlos Ruiz          carlos@email.com
Luis Gomez           luis@email.com
Maria Lopez          maria@email.com
